In [ ]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
from datetime import datetime, timezone
import json
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

In [6]:
file_path = Path("/Users/mnatali/Projects/sentiment_analysis/brightdata_across_social_media_analysis/brightdata_social_exports/reddit_datacenters_posts.json")

with file_path.open("r", encoding="utf-8") as f:
    reddit_posts = json.load(f)

In [7]:
all_post_ids = []

env_post_ids = []
infr_post_ids = []
housing_post_ids = []
econ_post_ids = []
life_qual_post_ids = []
aesth_post_ids = []
gov_post_ids = []
not_useful_post_ids = []

environmental_keywords = ["environment", "pollution", "emissions", "diesel", "fumes", "light pollution", "water", "drought", "water consumption",  "waste", "carbon", "climate"]
infrastructure_utilities_keywords = ["utilities", "utility", "electric", "power", "grid", "blackout", "substation", "reliable", "reliability", "capacity", "infrastructure", "generator", "water"]
housing_keywords = ["housing", "house", "home", "rent", "property", "real estate", "house price", "housing price", "housing cost", "land cost", "land price", "gentrification"]
economic_keywords = ["job", "work", "revenue", "tax", "money", "rich", "salary", "economy", "investment", "business"]
life_quality_keywords = ["resident", "fumes", "noise", "loud", "quiet", "vibration", "light pollution", "sound", "smell", "traffic"]
aesthetics_keywords = ["landscape", "visual impact",  "aesthetics", "looks", "architecture", "design", "facade", "industrial", "windows", "dark"]
government_keywords = ["zoning", "zone", "planning", "approval", "permit", "city council", "bill", "mayor", "governor", "government"]

matched_posts = 0

for red_post in reddit_posts:
    post_id = red_post["post_id"]
    all_post_ids.append(post_id)
    text = (red_post["description"])

    matched = False

    if any(kw in text for kw in environmental_keywords):
        env_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in infrastructure_utilities_keywords):
        infr_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in housing_keywords):
        housing_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in economic_keywords):
        econ_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in life_quality_keywords):
        life_qual_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in aesthetics_keywords):
        aesth_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in government_keywords):
        gov_post_ids.append(post_id)
        matched = True
    if not matched:
        not_useful_post_ids.append(post_id)
    else:
        matched_posts += 1

print("Total posts:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))

posts_by_id = {post["post_id"]: post for post in reddit_posts}

Total posts: 32900
Total posts with a theme: 30398
Found environmental posts: 9483
Found infrastructure posts: 20651
Found housing posts: 16309
Found economic posts: 23484
Found life quality posts: 13207
Found aesthetic posts: 10974
Found government posts: 12652


In [ ]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_num_comments = []
    post_upvotes = []
    post_subreddits = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["description"])
        post_dates.append(post["date_posted"])
        post_num_comments.append(post["num_comments"])
        post_upvotes.append(post["num_upvotes"])
        post_subreddits.append(post["community_name"])


    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["number of comments"] = post_num_comments
    df1["upvotes"] = post_upvotes
    df1["subreddit"] = post_subreddits


    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    if theme == env_post_ids:
        df1["environment"] = True
    if theme == infr_post_ids:
        df1["infrastructure"] = True
    if theme == housing_post_ids:
        df1["housing"] = True
    if theme == econ_post_ids:
        df1["economy"] = True
    if theme == life_qual_post_ids:
        df1["life quality"] = True
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
    if theme == gov_post_ids:
        df1["government"] = True
    posts = pd.concat([posts, df1], ignore_index=True)


posts = posts.astype({
    "ids": "string",
    "text": "string",
    "date": "string",
    "upvotes": "int64",
    "number of comments": "int64",
    "subreddit": "string",
    "environment": "bool",
    "infrastructure": "bool",
    "housing": "bool",
    "economy": "bool",
    "life quality": "bool",
    "aesthetics": "bool",
    "government": "bool",
})

len(posts['ids'])

106760

In [ ]:
def remove_links(text):
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

sample_string = "The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...  [https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#](https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#)  A group of residents from the [Save Bren Mar From Data Center](https://sites.google.com/view/savebrenmar/home) group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial properties.  [https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board](https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board)Fairfax County considers enhancing data center guidelines"
result = remove_links(sample_string)
print(result)

posts["text"] = posts["text"].apply(remove_links)

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [10]:
grouping_cols = ["ids", "text", "date"]
posts = posts.groupby(grouping_cols, as_index=False).max()
theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
len(posts['ids'])

30398

In [ ]:
company_terms = ["AWS","Amazon","Google","Microsoft","Azure","Meta","Oracle","Equinix","Digital Realty","IBM","Facebook","Apple","QTS","Vantage","CyrusOne","CoreSite"]

for c in company_terms:
    posts[c] = posts["text"].str.contains(rf'\b{re.escape(c)}\b', case=False, na=False)

posts_with_company = posts[posts[company_terms].any(axis=1)]
posts_with_company.head()

In [11]:
posts.head()

,ids,text,date,upvotes,number of comments,subreddit,environment,infrastructure,housing,economy,life quality,aesthetics,government
0,t3_100yio2,"Being in Telecom since 1985, this is something...",2023-01-02T00:42:27.229Z,4,16,replika,False,False,True,True,False,False,False
1,t3_10254az,"According to a research report "" Neuromorphic ...",2023-01-03T10:55:58.213Z,1,0,MnMResearchStudies,False,True,True,True,True,True,True
2,t3_107pxv6,I have been working with PA support for a few ...,2023-01-09T21:03:30.581Z,11,21,paloaltonetworks,False,False,True,True,True,False,True
3,t3_107ubhz,We're looking to use more feature flags in our...,2023-01-09T23:49:15.993Z,10,29,webdev,False,True,True,False,False,False,False
4,t3_1087m2o,Riot wants to boast inclusion through characte...,2023-01-10T11:31:02.422Z,2928,169,VALORANT,True,True,False,True,False,True,False


In [12]:
# def convert_dates(utc_date):
#     date = datetime.fromtimestamp(utc_date, tz=timezone.utc)
#     return date

# posts['date'] = posts['date'].apply(convert_dates)
# posts.head(10)

In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    padding=True,
    max_length=512,
    top_k=None
)

tokenizer = classifier.tokenizer

def chunk_text(text, max_tokens=450):
    """
    Split one post into chunks small enough for RoBERTa.
    450 leaves room for special tokens and avoids hitting the 512 limit.
    """
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) == 0:
        return [""]

    chunks = []

    for i in range(0, len(token_ids), max_tokens):
        chunk_ids = token_ids[i:i + max_tokens]
        chunk = tokenizer.decode(chunk_ids, skip_special_tokens=True)
        chunks.append(chunk)

    return chunks


def aggregate_chunk_results(chunk_results):
    scores = {
        "negative": [],
        "neutral": [],
        "positive": []
    }

    for result in chunk_results:
        for item in result:
            label = item["label"].lower()
            scores[label].append(item["score"])

    avg_scores = {
        label: float(np.mean(values)) if values else 0.0
        for label, values in scores.items()
    }

    final_label = max(avg_scores, key=avg_scores.get)

    return {
        "label": final_label,
        "score": avg_scores[final_label],
        "scores": avg_scores,
        "n_chunks": len(chunk_results)
    }

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [ ]:
texts= posts["text"].fillna("").astype(str).tolist()
all_post_results = []

for text in tqdm(texts):
    chunks = chunk_text(text)

    chunk_results = classifier(
        chunks,
        truncation=True,
        padding=True,
        max_length=512,
        batch_size=32
    )

    post_result = aggregate_chunk_results(chunk_results)
    all_post_results.append(post_result)

posts["roberta"] = all_post_results

30,000 posts took around 11 minutes

In [ ]:
posts["name"] = posts["roberta"].apply(lambda x: x["label"])
posts["label"] = posts["name"]

posts["degree"] = posts["roberta"].apply(lambda x: x["score"])
posts = posts.drop(["roberta", "name"], axis=1)
posts.head(15)

,ids,text,date,upvotes,number of comments,subreddit,environment,infrastructure,housing,economy,life quality,aesthetics,government,label,degree,year
0,t3_100yio2,"Being in Telecom since 1985, this is something...",2023-01-02T00:42:27.229Z,4,16,replika,False,False,True,True,False,False,False,neutral,0.710796,2023
1,t3_10254az,"According to a research report "" Neuromorphic ...",2023-01-03T10:55:58.213Z,1,0,MnMResearchStudies,False,True,True,True,True,True,True,neutral,0.824037,2023
2,t3_107pxv6,I have been working with PA support for a few ...,2023-01-09T21:03:30.581Z,11,21,paloaltonetworks,False,False,True,True,True,False,True,negative,0.627526,2023
3,t3_107ubhz,We're looking to use more feature flags in our...,2023-01-09T23:49:15.993Z,10,29,webdev,False,True,True,False,False,False,False,neutral,0.728118,2023
4,t3_1087m2o,Riot wants to boast inclusion through characte...,2023-01-10T11:31:02.422Z,2928,169,VALORANT,True,True,False,True,False,True,False,negative,0.549073,2023
5,t3_108o9iz,"I have recently become aware of Filen.io, an e...",2023-01-10T23:11:20.203Z,16,6,privacy,False,False,True,False,True,False,False,positive,0.581461,2023
6,t3_108y28m,Here’s the list of pre-filed bills for the 202...,2023-01-11T06:46:40.513Z,2,0,WaldorfMD,True,False,True,True,True,False,True,neutral,0.937541,2023
7,t3_10fkhem,"Hello, Is there a way to show a custom text an...",2023-01-18T22:31:41.364Z,3,3,jira,False,False,False,True,False,False,False,neutral,0.747904,2023
8,t3_10gfgpm,Source: https://twitter.com/Tristan0x/status/1...,2023-01-19T22:34:25.776Z,11,2,solana,False,False,False,True,True,False,False,neutral,0.753392,2023
9,t3_10gw3b0,Hi All We are currently developing our own web...,2023-01-20T12:46:49.812Z,2,25,datacenter,False,True,True,True,False,False,False,neutral,0.525674,2023


In [23]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    pos = theme_posts.loc[theme_posts["label"] == "positive", "degree"].sum()
    neg = theme_posts.loc[theme_posts["label"] == "negative", "degree"].sum()
    total = theme_posts["degree"].sum()
    return (pos-neg)/total

print("Average sentiment of environmental posts:", avg_sentiment_calculation("environment"))
print("Average sentiment of infrastructural posts:", avg_sentiment_calculation("infrastructure"))
print("Average sentiment of housing-related posts:", avg_sentiment_calculation("housing"))
print("Average sentiment of economic posts:", avg_sentiment_calculation("economy"))
print("Average sentiment of life-quality-related posts:", avg_sentiment_calculation("life quality"))
print("Average sentiment of aesthetics-related posts:", avg_sentiment_calculation("aesthetics"))
print("Average sentiment of governmental posts:", avg_sentiment_calculation("government"))

Average sentiment of environmental posts: 0.04881908653237448
Average sentiment of infrastructural posts: 0.12428437477648148
Average sentiment of housing-related posts: 0.10624754354471655
Average sentiment of economic posts: 0.13435785850590581
Average sentiment of life-quality-related posts: 0.11142615112555028
Average sentiment of aesthetics-related posts: 0.16731839277583144
Average sentiment of governmental posts: 0.10611726017885022


In [ ]:
## total average sentiment calculation

pos = posts.loc[posts["label"] == "positive", "degree"].sum()
neg = posts.loc[posts["label"] == "negative", "degree"].sum()
total = posts["degree"].sum()
print("Average sentiment for all of reddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  0.1027072042577493


In [25]:
pos = posts.loc[posts["label"] == "positive"]
neg = posts.loc[posts["label"] == "negative"]
neutral = posts.loc[posts["label"] == "neutral"]

print("Percent of posts that are neutral:", len(neutral)/len(posts))
print("Percent of posts that are negative:", len(neg)/len(posts))
print("Percent of posts that are positive:", len(pos)/len(posts))

really_pos = pos.loc[posts["degree"] > 0.70]
really_neg = neg.loc[posts["degree"] > 0.70]

if len(neg) != 0:
    print("Percent of negative posts that are really negative:", len(really_neg)/len(neg))
if len(pos) != 0:
    print("Percent of positive posts that are really positive:", len(really_pos)/len(pos))

Percent of posts that are neutral: 0.6649450621751432
Percent of posts that are negative: 0.12099480228962432
Percent of posts that are positive: 0.2140601355352326
Percent of negative posts that are really negative: 0.367047308319739
Percent of positive posts that are really positive: 0.508221914860919


In [ ]:
def avg_sentiment_calculation(dataset):
    pos = dataset.loc[dataset["label"] == "positive", "degree"].sum()
    neg = dataset.loc[dataset["label"] == "negative", "degree"].sum()
    total = dataset["degree"].sum()
    if total != 0:
        return (pos-neg)/total
    else:
        return 0

posts['year'] = pd.to_datetime(posts['date']).dt.year

year_datasets = {year: posts[posts['year'] == year] for year in range(2010, 2027)}


for i in range(2010, 2027):
    print(f"Number of posts from {i}: ", len(year_datasets[i]))
    if(avg_sentiment_calculation(year_datasets[i]) != 0):
        print(f"Average sentiment for reddit posts from {i}: ", avg_sentiment_calculation(year_datasets[i]))

Average sentiment for reddit posts from 2010:  0
Average sentiment for reddit posts from 2011:  0.0
Average sentiment for reddit posts from 2012:  0.31620337102136914
Average sentiment for reddit posts from 2013:  0.05847390763647159
Average sentiment for reddit posts from 2014:  -0.007912258250271326
Average sentiment for reddit posts from 2015:  0.2985466281077001
Average sentiment for reddit posts from 2016:  0.21704295956216904
Average sentiment for reddit posts from 2017:  0.24457473691021023
Average sentiment for reddit posts from 2018:  -0.013405082781437797
Average sentiment for reddit posts from 2019:  0.15208498775705584
Average sentiment for reddit posts from 2020:  0.10119542790490123
Average sentiment for reddit posts from 2021:  0.2221979700638281
Average sentiment for reddit posts from 2022:  0.14287861081749664
Average sentiment for reddit posts from 2023:  0.1471453881444923
Average sentiment for reddit posts from 2024:  0.2445741717800335
Average sentiment for reddit 

In [ ]:
def sentiment_by_company(company1, company2=None):
    company_posts = posts[
        (posts[company1] == True) | (posts[company2] == True) if company2 else (posts[company1] == True)
    ]
    return avg_sentiment_calculation(company_posts)

print("Number of Amazon-related posts:", len(posts[posts["AWS"] == True]) + len(posts[posts["Amazon"] == True]))
print("Average sentiment of AWS-related posts:", sentiment_by_company("AWS", "Amazon"))
print("Number of Google-related posts:", len(posts[posts["Google"] == True]))
print("Average sentiment of Google-related posts:", sentiment_by_company("Google"))
print("Number of Microsoft-related posts:", len(posts[posts["Microsoft"] == True]) + len(posts[posts["Azure"] == True]))
print("Average sentiment of Microsoft-related posts:", sentiment_by_company("Microsoft", "Azure"))
print("Number of Meta-related posts:", len(posts[posts["Meta"] == True]) + len(posts[posts["Facebook"] == True]))
print("Average sentiment of Meta-related posts:", sentiment_by_company("Meta"))
print("Number of Oracle-related posts:", len(posts[posts["Oracle"] == True]))
print("Average sentiment of Oracle-related posts:", sentiment_by_company("Oracle"))
print("Number of Equinix-related posts:", len(posts[posts["Equinix"] == True]))
print("Average sentiment of Equinix-related posts:", sentiment_by_company("Equinix"))
print("Number of Digital Realty-related posts:", len(posts[posts["Digital Realty"] == True]))
print("Average sentiment of Digital Realty-related posts:", sentiment_by_company("Digital Realty"))
print("Number of IBM-related posts:", len(posts[posts["IBM"] == True]))
print("Average sentiment of IBM-related posts:", sentiment_by_company("IBM"))
print("Number of Apple-related posts:", len(posts[posts["Apple"] == True]))
print("Average sentiment of Apple-related posts:", sentiment_by_company("Apple"))
print("Number of QTS-related posts:", len(posts[posts["QTS"] == True]))
print("Average sentiment of QTS-related posts:", sentiment_by_company("QTS"))
print("Number of Vantage-related posts:", len(posts[posts["Vantage"] == True]))
print("Average sentiment of Vantage-related posts:", sentiment_by_company("Vantage"))
print("Number of CyrusOne-related posts:", len(posts[posts["CyrusOne"] == True]))
print("Average sentiment of CyrusOne-related posts:", sentiment_by_company("CyrusOne"))
print("Number of CoreSite-related posts:", len(posts[posts["CoreSite"] == True]))
print("Average sentiment of CoreSite-related posts:", sentiment_by_company("CoreSite"))

In [ ]:
def analysis_of_viral_posts(min):
    viral_posts = posts.loc[posts["upvotes"] >= min]
    print("What's considered viral: posts with over", min, "upvotes")
    print("Number of viral posts:", len(viral_posts))
    print("Average sentiment of viral posts:", avg_sentiment_calculation(viral_posts))
    print("Average sentiment of non-viral posts:", avg_sentiment_calculation(posts.loc[posts["upvotes"] < min]))
    print("Amount percent of viral posts that are polar (degree) > 0.70: ", len(viral_posts.loc[viral_posts["degree"] > 0.70])/len(viral_posts))

analysis_of_viral_posts(100)
print("\n")
analysis_of_viral_posts(1000)